<a href="https://colab.research.google.com/github/v3rmxni7/fine_tuning_SLM/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Qwen2.5-0.5B-Instruct with QLoRA

**Task:** Structured Information Extraction from unstructured text

**Run this notebook on Google Colab with a T4 GPU runtime.**

Steps:
1. Install dependencies
2. Upload dataset
3. Train with QLoRA
4. Evaluate base vs fine-tuned
5. Download LoRA adapter

In [1]:
# Step 0: Install dependencies
!pip install -q torch transformers peft bitsandbytes trl datasets accelerate scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.5 MB/s eta 0:00:00


In [2]:
# Step 0.5: Clone repo (if running on Colab)
import os
if not os.path.exists('fine_tuning_SLM'):
    !git clone https://github.com/v3rmxni7/fine_tuning_SLM.git
    os.chdir('fine_tuning_SLM')
else:
    os.chdir('fine_tuning_SLM')

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Cloning into 'fine_tuning_SLM'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 55 (delta 20), reused 48 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 131.81 KiB | 661.00 KiB/s, done.
Resolving deltas: 100% (20/20), done.
CUDA available: True
GPU: Tesla T4


In [3]:
# Step 1: Generate dataset (if not already generated)
if not os.path.exists('data/processed/train.json'):
    !python data/generate_dataset.py
else:
    print('Dataset already exists.')

import json
with open('data/processed/train.json') as f:
    train_data = json.load(f)
print(f'Training samples: {len(train_data)}')
print(f'Sample: {json.dumps(train_data[0], indent=2)[:300]}')

Dataset already exists.
Training samples: 800
Sample: {
  "instruction": "Extract structured information from the following text and return valid JSON.",
  "input": "Richard Sanchez works as a Platform Engineer. With 7+ years of professional experience, they specialize in Spring Boot, SQL, Git and Airflow. Education: Master's in AI from UC Berkeley.",



In [4]:
# Step 2: Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Model loaded.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [5]:
# Step 3: Attach LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    task_type='CAUSAL_LM',
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [6]:
# Step 4: Prepare dataset for training
from datasets import Dataset
from src.formatting import format_chat_messages

def prepare_data(samples):
    formatted = []
    for sample in samples:
        category = sample.get('category', 'resume')
        messages = format_chat_messages(sample, category)
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({'text': text})
    return Dataset.from_list(formatted)

with open('data/processed/train.json') as f:
    train_samples = json.load(f)
with open('data/processed/val.json') as f:
    val_samples = json.load(f)

train_dataset = prepare_data(train_samples)
val_dataset = prepare_data(val_samples)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
print(f'\nSample formatted text:\n{train_dataset[0]["text"][:500]}')

Train: 800, Val: 100

Sample formatted text:
<|im_start|>system
You are an information extraction system.

Rules:
- Output STRICT JSON only. No explanation, no markdown, no extra text.
- Follow the schema exactly. No extra fields.
- If a field is not present in the input, return null for that field.

Schema:
{
  "name": string,
  "role": string,
  "experience": string,
  "skills": list of strings,
  "education": string
}<|im_end|>
<|im_start|>user
Extract structured information from the following text and return valid JSON.

Richard Sanche


In [7]:
# Step 5: Train
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir='outputs',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    fp16=False,
    bf16=False,
    report_to='none',
    remove_unused_columns=False,
    optim='paged_adamw_8bit',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print('Starting training...')
trainer.train()

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss,Validation Loss
50,0.455105,0.428466
100,0.338151,0.332366
150,0.303453,0.312406


TrainOutput(global_step=150, training_loss=0.5850302155812581, metrics={'train_runtime': 834.5445, 'train_samples_per_second': 2.876, 'train_steps_per_second': 0.18, 'total_flos': 1229394476006400.0, 'train_loss': 0.5850302155812581})

In [8]:
# Step 6: Save LoRA adapter
ADAPTER_DIR = 'outputs/lora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

# List saved files
for f in os.listdir(ADAPTER_DIR):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024 / 1024
    print(f'  {f}: {size_mb:.2f} MB')

Adapter saved to outputs/lora_adapter
  adapter_config.json: 0.00 MB
  README.md: 0.00 MB
  adapter_model.safetensors: 4.15 MB
  tokenizer.json: 10.89 MB
  chat_template.jinja: 0.00 MB
  tokenizer_config.json: 0.00 MB


In [9]:
# Step 7: Quick inference test (base vs fine-tuned)
from peft import PeftModel

test_input = 'Sarah Chen is a senior data engineer with 5 years of experience in Apache Spark, Python, and AWS. She holds a Masters in CS from Stanford.'

# Format prompt
from src.formatting import format_inference_messages
messages = format_inference_messages(test_input, 'resume')

def generate(model, messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    input_len = inputs['input_ids'].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

# Fine-tuned output
ft_output = generate(model, messages)
print('Fine-tuned output:')
print(ft_output)

# Base model output (reload without adapter)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_output = generate(base_model, messages)
print('\nBase model output:')
print(base_output)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Fine-tuned output:
{"name": "Sarah Chen", "role": "Senior Data Engineer", "experience": "5 years", "skills": ["Apache Spark", "Python", "AWS"], "education": "Masters in CS from Stanford"}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Base model output:
{
  "name": "Sarah Chen",
  "role": "Senior Data Engineer",
  "experience": [
    "5 years"
  ],
  "skills": [
    "Apache Spark",
    "Python",
    "AWS"
  ],
  "education": "Masters in CS from Stanford"
}


In [10]:
# Step 8: Run full evaluation on test set
from src.inference import InferencePipeline

with open('data/processed/test.json') as f:
    test_data = json.load(f)

print(f'Evaluating on {len(test_data)} test samples...')

# 8a: Base model (no guardrails)
print('\n--- Evaluating Base Model ---')
base_pipeline = InferencePipeline(use_finetuned=False, use_guardrails=False)
for sample in test_data:
    base_pipeline.extract(sample['input'], category=sample['category'])
base_pipeline.save_logs('evaluation/logs_base.json')

# 8b: Fine-tuned (no guardrails)
print('\n--- Evaluating Fine-tuned Model ---')
ft_pipeline = InferencePipeline(use_finetuned=True, use_guardrails=False)
for sample in test_data:
    ft_pipeline.extract(sample['input'], category=sample['category'])
ft_pipeline.save_logs('evaluation/logs_finetuned.json')

# 8c: Fine-tuned + Guardrails
print('\n--- Evaluating Fine-tuned + Guardrails ---')
ftg_pipeline = InferencePipeline(use_finetuned=True, use_guardrails=True)
for sample in test_data:
    ftg_pipeline.extract(sample['input'], category=sample['category'])
ftg_pipeline.save_logs('evaluation/logs_finetuned_guardrails.json')

Evaluating on 100 test samples...

--- Evaluating Base Model ---
Loading base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model ready.
Saved 100 inference logs to evaluation/logs_base.json

--- Evaluating Fine-tuned Model ---
Loading fine-tuned model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LoRA adapter from outputs/lora_adapter...
Model ready.
Saved 100 inference logs to evaluation/logs_finetuned.json

--- Evaluating Fine-tuned + Guardrails ---
Loading fine-tuned model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LoRA adapter from outputs/lora_adapter...
Model ready.
Saved 100 inference logs to evaluation/logs_finetuned_guardrails.json


In [11]:
# Step 9: Print evaluation results
from evaluation.eval import run_evaluation, print_comparison_table, save_results, load_test_data

test_data = load_test_data()
results = []

for log_path, label in [
    ('evaluation/logs_base.json', 'Base Model'),
    ('evaluation/logs_finetuned.json', 'Fine-tuned'),
    ('evaluation/logs_finetuned_guardrails.json', 'Fine-tuned + Guardrails'),
]:
    if os.path.exists(log_path):
        metrics = run_evaluation(log_path, label, test_data)
        results.append(metrics)

print_comparison_table(results)
save_results(results)


EVALUATION RESULTS: 3-Way Comparison
Metric                   Base Model          Fine-tuned          Fine-tuned + Guardrails
--------------------------------------------------------------------------------
JSON Validity Rate       0.0400              1.0000              1.0000              
Exact Match Accuracy     0.0000              0.9100              0.9100              
Field Precision          0.7000              0.9843              0.9843              
Field Recall             0.6500              0.9843              0.9843              
Field F1                 0.6722              0.9843              0.9843              
Failure Rate             0.9600              0.0000              0.0000              
Retry Rate               0.0000              0.0000              0.0000              
Avg Retries/Sample       0.0000              0.0000              0.0000              
Retry Success Rate       0                   0                   0                   

Error Taxonomy:
-

In [12]:
# Step 10: Download adapter (for Colab)
# Zip the adapter for easy download
!zip -r lora_adapter.zip outputs/lora_adapter/
print('Download lora_adapter.zip from the file browser on the left.')

# Optional: Push to HuggingFace
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# model.push_to_hub('your-username/qwen2.5-0.5b-extraction-lora')
# tokenizer.push_to_hub('your-username/qwen2.5-0.5b-extraction-lora')

  adding: outputs/lora_adapter/ (stored 0%)
  adding: outputs/lora_adapter/adapter_config.json (deflated 57%)
  adding: outputs/lora_adapter/README.md (deflated 65%)
  adding: outputs/lora_adapter/adapter_model.safetensors (deflated 21%)
  adding: outputs/lora_adapter/tokenizer.json (deflated 81%)
  adding: outputs/lora_adapter/chat_template.jinja (deflated 71%)
  adding: outputs/lora_adapter/tokenizer_config.json (deflated 59%)
Download lora_adapter.zip from the file browser on the left.
